<a href="https://colab.research.google.com/github/iarabertopena/ml-learning-lab/blob/main/stream_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Bibliotecas

In [1]:
%%writefile requirements.txt
numpy
pandas
scipy
matplotlib
seaborn
capymoa

Overwriting requirements.txt


In [2]:
!pip install -r requirements.txt

### Imports

In [3]:
from capymoa.classifier import (
HoeffdingAdaptiveTree,
AdaptiveRandomForestClassifier,
SAMkNN,
StreamingGradientBoostedTrees,
StreamingRandomPatches
)

from capymoa.evaluation import (
ClassificationEvaluator,
ClassificationWindowedEvaluator
)

### Datasets

In [4]:
from capymoa.datasets import Sensor
stream = Sensor()
stream.next_instance().x

sensor.arff: 15.0MB [00:04, 3.15MB/s]                            


array([58.     , 19.7336 , 37.0933 , 71.76   ,  2.69964])

In [5]:
from capymoa.datasets import Electricity
stream = Electricity()
stream.next_instance().x

electricity.arff: 704kB [00:01, 447kB/s]                    


array([0.      , 1.      , 0.      , 0.056443, 0.439155, 0.003467,
       0.422915, 0.414912])

In [6]:
from capymoa.datasets import Hyper100k
stream = Hyper100k()
stream.next_instance().x

Hyper100k.arff: 8.59MB [00:02, 3.55MB/s]                            


array([0.39717434, 0.34751803, 0.29405703, 0.50648363, 0.11596709,
       0.77053588, 0.65989271, 0.15674689, 0.37820205, 0.13976268])

In [7]:
from capymoa.datasets import RTG_2abrupt
stream = RTG_2abrupt()
stream.next_instance().x

RTG_2abrupt.arff: 25.3MB [00:05, 4.80MB/s]                            


array([0.73087819, 0.41008081, 0.20771484, 0.33271706, 0.96775591,
       0.00611718, 0.9637048 , 0.93986539, 0.94719492, 0.93708215,
       0.39717434, 0.34751803, 0.29405703, 0.50648363, 0.11596709,
       0.77053588, 0.65989271, 0.15674689, 0.37820205, 0.13976268,
       0.69494798, 0.80522777, 0.00502518, 0.52313516, 0.74398449,
       0.1420227 , 0.4817283 , 0.54455481, 0.57710026, 0.20491355])

In [8]:
from capymoa.datasets import CovtypeTiny
stream = CovtypeTiny()
stream.next_instance().x

covtype_n1000.arff: 24.0kB [00:02, 8.66kB/s]                            


array([2.596e+03, 5.100e+01, 3.000e+00, 2.580e+02, 0.000e+00, 5.100e+02,
       2.210e+02, 2.320e+02, 1.480e+02, 6.279e+03, 1.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00])

In [9]:
datasets = {
    "Sensor": Sensor,
    "Electricity": Electricity,
    "Hyper100k": Hyper100k,
    "RTG_2abrupt": RTG_2abrupt,
    "CovtypeTiny": CovtypeTiny
}

### Métricas de avaliação

In [16]:
def get_evaluators():

    return {
        "global": ClassificationEvaluator(),
        "window": ClassificationWindowedEvaluator(window_size=1000),
    }

In [17]:
def extract_metrics(evaluator):

    results = {}

    for ev in evaluator.evaluators:

        metrics = ev.get_metrics()
        name = type(ev).__name__

        results[name] = {
            "accuracy": metrics.get("accuracy"),
            "kappa": metrics.get("kappa")
        }

    return results

In [12]:
# Accuracy:
# Proporção de instâncias corretamente classificadas.
# Mede o desempenho geral do modelo ao longo do stream (pode ser influenciada por desbalanceamento entre classes).

# Kappa (Cohen’s Kappa):
# Mede o desempenho do modelo em relação ao acaso, considerando a distribuição das classes.
# Mais robusta que a accuracy em cenários desbalanceados.
# Valores:
#   1 → perfeito
#   0 → equivalente ao acaso
#   < 0 → pior que o acaso

# Accuracy (Sliding Window):
# Accuracy calculada apenas nas últimas N instâncias, refletindo o desempenho recente do modelo.
# Útil para detectar concept drift (quanto menor a janela, mais sensível a mudanças rápidas).

# Kappa (Sliding Window):
# Versão do Kappa considerando apenas a janela recente.
# Avalia a qualidade das previsões atuais em relação ao acaso, sendo + informativo que accuracy em streams com drift.

### Definindo os modelos

In [13]:
def get_models():
    return {
        "HAT": HoeffdingAdaptiveTree(),
        "ARF": AdaptiveRandomForestClassifier(),
        "SAMkNN": SAMkNN(),
        "SGBT": StreamingGradientBoostedTrees(),
        "SRP": StreamingRandomPatches()
    }

In [14]:
# HAT (Hoeffding Adaptive Tree):
# Árvore de decisão incremental baseada no limite de Hoeffding que aprende a partir de dados em fluxo (sem precisar armazenar tudo).
# Possui mecanismos de adaptação a concept drift, substituindo ramos da árvore quando detecta mudança no comportamento dos dados.

# ARF (Adaptive Random Forest):
# Ensemble de várias Hoeffding Trees, onde cada árvore aprende em subconjuntos diferentes dos dados (bagging online).
# Usa detectores de drift para substituir modelos ruins.

# SAMkNN (Self-Adjusting Memory kNN):
# Versão adaptativa do kNN para data streams, que mantém uma memória de curto e longo prazo.
# Ajusta dinamicamente quais dados usar para prever, lidando bem com concept drift ao esquecer padrões antigos.

# SGBT (Streaming Gradient Boosted Trees):
# Versão online do Gradient Boosting que combina várias árvores sequencialmente corrigindo erros anteriores.
# Aprende de forma incremental em fluxo de dados, podendo capturar relações complexas (mas é mais custoso computacionalmente).

# SRP (Streaming Random Patches):
# Ensemble que combina amostragem de instâncias e atributos, onde cada modelo vê um subconjunto diferente dos dados e features.
# Similar ao Random Forest, mas adaptado para data streams, com bom equilíbrio entre diversidade e desempenho.

### Loop principal

In [15]:
def run_experiments():

  all_results = {}

  for dataset_name in datasets:
    print(f"\nDataset: {dataset_name}")

    dataset_results = {}

    for name, model in get_models().items():
      print(f"\nRodando modelo: {name}")

      # criar stream
      stream = dataset_name()

      evaluators_dict = get_evaluators()

      evaluator = PrequentialEvaluation(
          stream = stream,
          model = model,
          evaluators = list(evaluators_dict.values())
      )

      evaluator.run()

      metrics = extract_metrics(evaluator, evaluators_dict.keys())

      dataset_results[name] = metrics

      all_results[dataset_name.__name__] = dataset_results

    return all_results